# Google Drive → Dailymotion (Kaggle)

Takes the private parts from a finished YouTube → Drive job and uploads them **one file at a time**. Each success is saved before the next download starts. The run stops at Dailymotion's standard cap (15 videos or 10 hours / 24 hours). Resume continues later.

When every part has a Dailymotion URL, the job is done and the admin can delete that named Drive folder. This worker does not delete Drive files.

**Secrets:** `API_BASE`, `COLAB_JOB_SECRET`, `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`, `DAILYMOTION_CLIENT_ID`, `DAILYMOTION_CLIENT_SECRET`, optional `DAILYMOTION_PROFILE_ID`.


In [ ]:
#!/usr/bin/env python3
"""Kaggle worker: upload one Google Drive part at a time to Dailymotion.

Successful uploads are recorded on the job. When every part in the named folder
has a Dailymotion URL, the admin is told they can delete that folder.
"""

from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

DM_MAX_FILE_BYTES = 4 * 1024 * 1024 * 1024
DM_MAX_FILE_SECONDS = 2 * 3600
DM_MAX_VIDEOS_PER_DAY = 15
DM_MAX_SECONDS_PER_DAY = 10 * 3600
DAILY_LIMIT_HINTS = (
    "quota",
    "too many",
    "rate limit",
    "daily limit",
    "upload limit",
    "exceeded your",
    "reached your",
)
PART_PATTERN = re.compile(r"^(.*) - Part (\d+)$")

def format_hms(seconds: float) -> str:
    total = max(0, int(seconds))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    return f"{minutes}m{secs:02d}s"

def log(message: str) -> None:
    print(message, flush=True)

def disk_status(path: Path) -> None:
    usage = shutil.disk_usage(path)
    log(f"💾 Disk: {usage.used / 1e9:.1f} GB used, {usage.free / 1e9:.1f} GB free")

def remove_path(path: str | Path | None) -> None:
    if not path:
        return
    target = Path(path)
    try:
        if target.is_dir():
            shutil.rmtree(target, ignore_errors=True)
        elif target.exists():
            target.unlink()
    except OSError as err:
        log(f"⚠️ Could not delete {target}: {err}")

def load_injected_secrets() -> dict[str, str]:
    raw = "__INJECTED_SECRETS_B64__"
    if not raw or raw.startswith("__"):
        return {}
    try:
        import base64
        import json as _json

        payload = _json.loads(base64.b64decode(raw).decode())
        if isinstance(payload, dict):
            return {str(key): str(value) for key, value in payload.items() if value}
    except Exception as err:
        log(f"Could not read injected settings: {err}")
    return {}


INJECTED_SECRETS = load_injected_secrets()


def get_secret(name: str, *aliases: str) -> str:
    keys = (name, *aliases)
    for key in keys:
        value = os.environ.get(key, "").strip()
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for key in keys:
            try:
                value = (client.get_secret(key) or "").strip()
            except Exception:
                value = ""
            if value:
                return value
    except Exception as err:
        log(f"Kaggle user secrets unavailable: {err}")
    for key in keys:
        value = str(INJECTED_SECRETS.get(key) or "").strip()
        if value:
            return value
    return ""

def api_headers(secret: str) -> dict[str, str]:
    return {"x-colab-secret": secret, "Content-Type": "application/json"}

def is_quota_like_text(text: str) -> bool:
    lower = text.lower()
    return any(hint in lower for hint in DAILY_LIMIT_HINTS)

def is_access_denied_text(text: str) -> bool:
    upper = text.upper()
    return "ACCESS_DENIED" in upper or "ACCESS_FORBIDDEN" in upper

def is_daily_limit_error(response) -> bool:
    if response.ok:
        return False
    return is_quota_like_text(response.text or "")

def log_dailymotion_response(step: str, response) -> None:
    body = (response.text or "").replace("\n", " ")[:400]
    log(f"   🌐 {step}: HTTP {response.status_code} {body}")

def extract_access_token(payload: object, raw: str) -> str:
    if not isinstance(payload, dict):
        raise RuntimeError(f"Dailymotion auth returned a non-object body: {raw[:500]}")
    token = payload.get("access_token") or payload.get("accessToken")
    if token:
        return str(token)
    raise RuntimeError(
        "Dailymotion auth returned no access_token. Password grant is gone — "
        "create a Private API key in Studio → Organization → API keys. "
        f"Response keys: {sorted(payload.keys())}. Body: {raw[:800]}"
    )

def dailymotion_login(credentials: dict) -> dict:
    import requests

    log("🔐 Authenticating with Dailymotion API v2 (client_credentials)...")
    response = requests.post(
        "https://oauth2.dailymotion.com/v2/token",
        data={
            "grant_type": "client_credentials",
            "client_id": credentials["client_id"],
            "client_secret": credentials["client_secret"],
            "scope": "video.manage account.read",
        },
        timeout=30,
    )
    raw = response.text
    try:
        payload = response.json()
    except ValueError as err:
        raise RuntimeError(
            f"Dailymotion auth returned non-JSON ({response.status_code}): {raw[:500]}"
        ) from err
    if response.status_code != 200:
        raise RuntimeError(f"Dailymotion auth failed: {response.status_code} {raw[:800]}")
    token = extract_access_token(payload, raw)
    profile_id = resolve_profile_id(token, credentials.get("profile_id"))
    log(f"✅ Logged in to Dailymotion (profile {profile_id})")
    return {"token": token, "profile_id": profile_id}

def list_accessible_profiles(token: str) -> list[dict[str, str]]:
    import requests

    res = requests.get(
        "https://api.dailymotion.com/v2/me",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if not res.ok:
        raise RuntimeError(
            "Could not load Dailymotion profiles from /v2/me. "
            f"{res.status_code} {res.text[:800]}"
        )
    payload = res.json()
    data = payload if isinstance(payload, dict) else {}
    raw = data.get("profiles") or []
    if isinstance(raw, dict):
        raw = raw.get("items") or raw.get("data") or []
    profiles: list[dict[str, str]] = []
    for profile in raw:
        if not isinstance(profile, dict):
            continue
        profile_id = profile.get("profile_id") or profile.get("id")
        if not profile_id:
            continue
        profiles.append(
            {
                "id": str(profile_id),
                "name": str(profile.get("name") or profile.get("display_name") or profile_id),
            }
        )
    return profiles

def resolve_profile_id(token: str, explicit: str | None) -> str:
    profiles = list_accessible_profiles(token)
    summary = ", ".join(f"{item['id']} ({item['name']})" for item in profiles) or "(none)"
    log(f"📺 Dailymotion profiles this API key can manage: {summary}")

    if explicit:
        if any(item["id"] == explicit for item in profiles):
            return explicit
        log(
            f"⚠️ DAILYMOTION_PROFILE_ID={explicit} is not in that list. "
            "It is often a public user/channel id, or the Private API key is limited to other channels."
        )
        if profiles:
            chosen = profiles[0]["id"]
            log(f"➡️ Using accessible profile {chosen} instead")
            return chosen
        raise RuntimeError(
            f"DAILYMOTION_PROFILE_ID={explicit} is not allowed for this Private API key. "
            "In Studio → Organization → API keys, include that channel on the key, "
            "or set the secret to a profile_id from GET /v2/me. "
            f"Accessible: {summary}"
        )

    if not profiles:
        raise RuntimeError(
            "This API key has no manageable Dailymotion profiles. "
            "In Studio → Organization → API keys, allow the channel you want to publish to, "
            "or set DAILYMOTION_PROFILE_ID to that channel's profile_id from GET /v2/me."
        )
    return profiles[0]["id"]

def upload_to_dailymotion(
    file_path: str,
    title: str,
    part_number: int | None,
    session: dict,
    channel: str,
    tags: str,
) -> dict:
    import requests
    from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor
    from tqdm import tqdm

    token = session["token"]
    profile_id = session["profile_id"]
    headers = {"Authorization": f"Bearer {token}"}
    log(f"   📤 Opening Dailymotion upload session for {Path(file_path).name} ({Path(file_path).stat().st_size / (1024 * 1024):.1f} MB)")
    res = requests.post(
        "https://api.dailymotion.com/v2/files/upload_sessions",
        headers=headers,
        timeout=30,
    )
    if not res.ok:
        log_dailymotion_response("upload_sessions", res)
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    upload_payload = res.json()
    upload_url = upload_payload.get("upload_url") or upload_payload.get("url")
    if not upload_url:
        raise RuntimeError(f"Dailymotion upload session had no URL: {res.text[:800]}")

    file_size = Path(file_path).stat().st_size
    pbar = tqdm(
        total=file_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=f"⬆️ {Path(file_path).name}",
    )
    start_time = time.time()

    def callback(monitor):
        try:
            pbar.n = monitor.bytes_read
            elapsed = time.time() - start_time
            if elapsed > 0 and monitor.bytes_read > 0:
                speed = monitor.bytes_read / elapsed
                remaining = max(file_size - monitor.bytes_read, 0)
                eta_sec = remaining / speed if speed > 0 else 0
                mins, secs = divmod(int(eta_sec), 60)
                pbar.set_postfix({"speed": f"{speed / 1024 / 1024:.2f} MB/s", "ETA": f"{mins}m {secs}s"})
            pbar.refresh()
        except Exception:
            pass

    with open(file_path, "rb") as handle:
        encoder = MultipartEncoder(fields={"file": (os.path.basename(file_path), handle, "video/mp4")})
        monitor = MultipartEncoderMonitor(encoder, callback)
        res = requests.post(
            upload_url,
            data=monitor,
            headers={"Content-Type": monitor.content_type},
            timeout=None,
        )
        if not res.ok:
            log_dailymotion_response("file upload", res)
        if is_daily_limit_error(res):
            pbar.close()
            raise DailyLimitReached(res.text)
        res.raise_for_status()
        uploaded = res.json()
        video_url = uploaded.get("url") or uploaded.get("file_url")
        if not video_url:
            raise RuntimeError(f"Dailymotion file upload returned no url: {res.text[:800]}")

    pbar.n = file_size
    pbar.refresh()
    pbar.close()

    final_title = f"{title} - Part {part_number}" if part_number else title
    final_title = final_title[:255]
    tag_list = [item.strip() for item in (tags or "").split(",") if item.strip()]
    log(f"   📝 Publishing '{final_title}'...")
    res = requests.post(
        f"https://api.dailymotion.com/v2/profiles/{profile_id}/videos",
        headers={**headers, "Content-Type": "application/json", "Accept": "application/json"},
        json={
            "title": final_title,
            "category": channel,
            "visibility": "public",
            "is_for_kids": False,
            "tags": tag_list,
            "source": {"file_url": video_url},
        },
        timeout=60,
    )
    if not res.ok:
        log_dailymotion_response("publish", res)
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    if res.status_code not in (200, 201):
        body = res.text[:800]
        if res.status_code in (401, 403) or is_access_denied_text(body):
            raise RuntimeError(
                f"Dailymotion denied publishing to profile {profile_id}. "
                "If earlier parts of this job already uploaded, this is often the daily "
                "cap (15 videos or 10 hours / 24h) rather than a bad API key. "
                "Otherwise check DAILYMOTION_PROFILE_ID against GET /v2/me. "
                f"{body}"
            )
        raise RuntimeError(f"Dailymotion publish failed ({res.status_code}): {body}")
    created = res.json()
    video_id = created.get("video_id") or created.get("id")
    if not video_id:
        raise RuntimeError(f"Dailymotion publish returned no video id: {res.text[:800]}")
    public_url = f"https://www.dailymotion.com/video/{video_id}"
    log(f"   ✅ {public_url}")
    return {"title": final_title, "videoId": video_id, "url": public_url}

def load_dailymotion_credentials() -> dict:
    creds = {
        "client_id": get_secret("DAILYMOTION_CLIENT_ID"),
        "client_secret": get_secret("DAILYMOTION_CLIENT_SECRET"),
        "profile_id": get_secret("DAILYMOTION_PROFILE_ID") or None,
    }
    if not creds["client_id"] or not creds["client_secret"]:
        raise SystemExit(
            "Missing DAILYMOTION_CLIENT_ID / DAILYMOTION_CLIENT_SECRET. "
            "Use a Private API key from Dailymotion Studio → Organization → API keys."
        )
    return creds

JOB_KIND = "drive_to_dailymotion"
WORK_DIR = Path("/kaggle/working/gdrive-dm")


class DailyLimitReached(Exception):
    pass


class JobCheckpoint:
    def __init__(self, urls: list[dict], next_part: int, resume_at_seconds: float) -> None:
        self.urls = list(urls)
        self.next_part = next_part
        self.resume_at_seconds = float(resume_at_seconds)

    @classmethod
    def from_job(cls, job: dict) -> "JobCheckpoint":
        return cls(
            urls=list(job.get("resultUrls") or []),
            next_part=int(job.get("startPartNumber") or 1),
            resume_at_seconds=float(job.get("resumeAtSeconds") or 0),
        )

    def payload(self) -> dict:
        return {
            "urls": self.urls,
            "nextStartPart": self.next_part,
            "resumeAtSeconds": int(self.resume_at_seconds),
        }


class PipelineStop(Exception):
    def __init__(self, message: str, checkpoint: JobCheckpoint, *, paused: bool) -> None:
        super().__init__(message)
        self.checkpoint = checkpoint
        self.paused = paused


class SessionQuota:
    def __init__(self) -> None:
        self.videos = 0
        self.seconds = 0.0

    @classmethod
    def from_job(cls, job: dict) -> "SessionQuota":
        from datetime import datetime, timezone

        quota = cls()
        now = datetime.now(timezone.utc)
        durations = {
            str(item.get("fileId")): float(item.get("durationSeconds") or 0)
            for item in (job.get("driveFiles") or [])
        }
        for item in job.get("resultUrls") or []:
            raw = item.get("uploadedAt")
            if not raw:
                continue
            try:
                stamp = datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
            except ValueError:
                continue
            if stamp.tzinfo is None:
                stamp = stamp.replace(tzinfo=timezone.utc)
            if (now - stamp.astimezone(timezone.utc)).total_seconds() > 24 * 3600:
                continue
            quota.videos += 1
            quota.seconds += durations.get(str(item.get("driveFileId") or ""), 0)
        if quota.videos or quota.seconds:
            log(
                f"📊 Counting {quota.videos} upload(s) / {format_hms(quota.seconds)} "
                "from the last 24 hours toward today's Dailymotion cap"
            )
        return quota

    def would_exceed(self, part_seconds: float) -> str | None:
        if part_seconds > DM_MAX_FILE_SECONDS + 1:
            return (
                f"Part length {format_hms(part_seconds)} exceeds Dailymotion's "
                "2-hour per-file limit."
            )
        if self.videos >= DM_MAX_VIDEOS_PER_DAY:
            return (
                f"Dailymotion standard daily video cap ({DM_MAX_VIDEOS_PER_DAY} videos / 24h) "
                "would be exceeded."
            )
        if self.seconds + part_seconds > DM_MAX_SECONDS_PER_DAY + 1:
            return (
                f"Dailymotion standard daily duration cap ({DM_MAX_SECONDS_PER_DAY // 3600} hours / 24h) "
                "would be exceeded."
            )
        return None

    def record(self, part_seconds: float) -> None:
        self.videos += 1
        self.seconds += part_seconds


def ensure_tools() -> None:
    log("📦 Installing Google Drive client, requests-toolbelt, tqdm...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "google-api-python-client",
            "requests-toolbelt",
            "tqdm",
        ],
        check=True,
    )


def claim_job(api_base: str, secret: str) -> dict | None:
    import requests

    res = requests.get(
        f"{api_base.rstrip('/')}/colab-jobs/pending",
        headers=api_headers(secret),
        params={"kind": JOB_KIND},
        timeout=30,
    )
    res.raise_for_status()
    return res.json().get("job")


def save_progress(api_base: str, secret: str, job_id: int, checkpoint: JobCheckpoint) -> None:
    import requests

    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/progress",
            headers=api_headers(secret),
            json=checkpoint.payload(),
            timeout=60,
        ).raise_for_status()
        log(
            f"💾 Checkpoint: {len(checkpoint.urls)} Dailymotion URL(s), next part {checkpoint.next_part}"
        )
    except Exception as err:
        log(f"⚠️ Could not save progress to backend: {err}")


def complete_job(api_base: str, secret: str, job_id: int, checkpoint: JobCheckpoint) -> None:
    import requests

    res = requests.post(
        f"{api_base.rstrip('/')}/colab-jobs/{job_id}/complete",
        headers=api_headers(secret),
        json=checkpoint.payload(),
        timeout=60,
    )
    res.raise_for_status()


def fail_job(
    api_base: str,
    secret: str,
    job_id: int,
    error_message: str,
    *,
    paused: bool = False,
    checkpoint: JobCheckpoint | None = None,
) -> None:
    import requests

    payload: dict = {"errorMessage": error_message[:4000], "paused": paused}
    if checkpoint is not None:
        payload.update(checkpoint.payload())
    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/fail",
            headers=api_headers(secret),
            json=payload,
            timeout=60,
        ).raise_for_status()
    except Exception as err:
        log(f"⚠️ Could not report failure to backend: {err}")


def raise_pause(reason: str, checkpoint: JobCheckpoint, cause: BaseException | None = None) -> None:
    message = (
        f"{reason} Saved {len(checkpoint.urls)} URL(s). "
        f"Resume at part {checkpoint.next_part} ({format_hms(checkpoint.resume_at_seconds)})."
    )
    if cause is None:
        raise PipelineStop(message, checkpoint, paused=True)
    raise PipelineStop(message, checkpoint, paused=True) from cause


def drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    client_id = get_secret("GDRIVE_CLIENT_ID")
    client_secret = get_secret("GDRIVE_CLIENT_SECRET")
    refresh_token = get_secret("GDRIVE_REFRESH_TOKEN")
    if not client_id or not client_secret or not refresh_token:
        raise RuntimeError(
            "Missing GDRIVE_CLIENT_ID, GDRIVE_CLIENT_SECRET, or GDRIVE_REFRESH_TOKEN."
        )
    creds = Credentials(
        token=None,
        refresh_token=refresh_token,
        client_id=client_id,
        client_secret=client_secret,
        token_uri="https://oauth2.googleapis.com/token",
        scopes=["https://www.googleapis.com/auth/drive"],
    )
    return build("drive", "v3", credentials=creds, cache_discovery=False)


def download_drive_file(service, file_id: str, dest: Path) -> None:
    from googleapiclient.http import MediaIoBaseDownload

    dest.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with dest.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                log(f"   ⬇️ Drive download {int(status.progress() * 100)}%")


def base_title(file_name: str, job: dict) -> str:
    stem = Path(file_name).stem
    match = PART_PATTERN.match(stem)
    if match:
        return match.group(1)
    custom = (job.get("customTitle") or "").strip()
    return custom or stem


def pending_files(job: dict, checkpoint: JobCheckpoint) -> list[dict]:
    done = {str(item.get("driveFileId")) for item in checkpoint.urls if item.get("driveFileId")}
    files = [item for item in (job.get("driveFiles") or []) if str(item.get("fileId")) not in done]
    files.sort(key=lambda item: int(item.get("partNumber") or 0))
    return files


def upload_pending(job: dict, service, session: dict, checkpoint: JobCheckpoint, quota: SessionQuota, on_progress) -> None:
    channel = job.get("dailymotionChannel") or "news"
    tags = job.get("dailymotionTags") or "youtube,upload"
    pending = pending_files(job, checkpoint)
    if not pending:
        log("ℹ️ Every Drive part already has a Dailymotion URL")
        return
    for item in pending:
        part = int(item.get("partNumber") or checkpoint.next_part)
        seconds = float(item.get("durationSeconds") or 0)
        reason = quota.would_exceed(seconds)
        if reason:
            raise_pause(reason, checkpoint)
        file_name = str(item.get("name") or f"part-{part}.mp4")
        dest = WORK_DIR / file_name
        log(f"📤 Part {part}: {file_name}")
        try:
            download_drive_file(service, str(item["fileId"]), dest)
            try:
                uploaded = upload_to_dailymotion(
                    str(dest),
                    base_title(file_name, job),
                    part,
                    session,
                    channel,
                    tags,
                )
            except DailyLimitReached as err:
                detail = str(err).replace("\n", " ")[:300]
                if not checkpoint.urls:
                    raise PipelineStop(
                        f"Dailymotion rejected the first upload before any part succeeded. {detail}",
                        checkpoint,
                        paused=False,
                    ) from err
                raise_pause(
                    "Dailymotion standard daily limit reached (15 videos or 10 hours / 24h). " + detail,
                    checkpoint,
                    err,
                )
            except Exception as err:
                if checkpoint.urls and is_access_denied_text(str(err)):
                    raise_pause(
                        "Dailymotion blocked the next publish after earlier parts succeeded. "
                        "This usually means the standard daily cap (15 videos or 10 hours / 24h).",
                        checkpoint,
                        err,
                    )
                raise PipelineStop(str(err), checkpoint, paused=False) from err
            from datetime import datetime, timezone

            uploaded["driveFileId"] = str(item["fileId"])
            uploaded["uploadedAt"] = datetime.now(timezone.utc).isoformat()
            checkpoint.urls.append(uploaded)
            checkpoint.next_part = part + 1
            checkpoint.resume_at_seconds += seconds
            quota.record(seconds)
            if on_progress:
                on_progress(checkpoint)
        finally:
            remove_path(dest)
            disk_status(WORK_DIR)


def run_pipeline(job: dict, service, session: dict, on_progress=None) -> JobCheckpoint:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = JobCheckpoint.from_job(job)
    quota = SessionQuota.from_job(job)
    if checkpoint.urls and quota.videos == 0 and quota.seconds == 0:
        checkpoint.resume_at_seconds = 0
    if quota.videos >= DM_MAX_VIDEOS_PER_DAY or quota.seconds >= DM_MAX_SECONDS_PER_DAY:
        raise_pause(
            "Dailymotion standard daily limit is still in effect for this job "
            "(15 videos or 10 hours / 24h). Wait for the window to reset, then Resume.",
            checkpoint,
        )
    if not (job.get("driveFolderId") or "").strip():
        raise RuntimeError("Job is missing the Google Drive folder of cut videos")
    try:
        upload_pending(job, service, session, checkpoint, quota, on_progress)
        if pending_files(job, checkpoint):
            raise_pause("Stopped before every Drive part was uploaded.", checkpoint)
        if not checkpoint.urls:
            raise RuntimeError("Pipeline finished with no Dailymotion URLs")
        return checkpoint
    except PipelineStop:
        raise
    except Exception as err:
        raise PipelineStop(str(err), checkpoint, paused=False) from err
    finally:
        remove_path(WORK_DIR)


def main() -> None:
    log("🧰 Worker build: gdrive-to-dailymotion")
    ensure_tools()
    api_base = get_secret("API_BASE")
    secret = get_secret("COLAB_JOB_SECRET", "JOB_SECRET")
    if not api_base or not secret:
        missing = []
        if not api_base:
            missing.append("API_BASE")
        if not secret:
            missing.append("COLAB_JOB_SECRET")
        raise SystemExit(
            "Missing "
            + " and ".join(missing)
            + ". Set API_PUBLIC_URL and COLAB_JOB_SECRET on the backend, then queue the job again."
        )

    session = dailymotion_login(load_dailymotion_credentials())
    service = drive_service()
    processed = 0
    while True:
        job = claim_job(api_base, secret)
        if job is None:
            break
        job_id = job.get("id")
        folder = job.get("driveFolderName") or job.get("driveFolderId")
        log(f"\n{'=' * 60}\n▶ Dailymotion job {job_id} — {folder}\n{'=' * 60}")
        try:
            def on_progress(checkpoint: JobCheckpoint) -> None:
                if job_id is not None:
                    save_progress(api_base, secret, job_id, checkpoint)

            checkpoint = run_pipeline(job, service, session, on_progress)
            if job_id is None:
                raise RuntimeError("Claimed job had no id")
            complete_job(api_base, secret, job_id, checkpoint)
            log(f"✅ Reported {len(checkpoint.urls)} URL(s) for job #{job_id}")
            for item in checkpoint.urls:
                log(f"   • {item['title']}: {item['url']}")
            folder_name = job.get("driveFolderName") or "the named Google Drive folder"
            log(
                f"🗑️ All parts are on Dailymotion. An admin can delete the Google Drive folder “{folder_name}”."
            )
        except PipelineStop as stop:
            kind = "paused" if stop.paused else "failed"
            log(f"{'⏸️' if stop.paused else '❌'} Job {job_id} {kind}: {stop}")
            if job_id is not None:
                fail_job(
                    api_base,
                    secret,
                    job_id,
                    str(stop),
                    paused=stop.paused,
                    checkpoint=stop.checkpoint,
                )
            if stop.paused:
                log("Stopping this run so the Dailymotion daily cap is not spent on another job.")
                break
        except Exception as err:
            log(f"❌ Job {job_id} failed: {err}")
            if job_id is not None:
                fail_job(api_base, secret, job_id, str(err), checkpoint=JobCheckpoint.from_job(job))
        processed += 1

    if processed == 0:
        log("No pending Drive → Dailymotion jobs")
    else:
        log(f"Done. Processed {processed} Dailymotion job(s).")


if __name__ == "__main__":
    main()
